In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/omarahmed05/cvcvcv/Omar Ahmed Abd El-Kader.pdf


In [2]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 90.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 89.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: requests

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=2000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
!pip install -U langchain-classic

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

full_name_schema = ResponseSchema(
    name="full_name",
    description="The complete legal name of the candidate, formatted as a single continuous string."
)

email_schema = ResponseSchema(
    name="email",
    description="The candidate's primary contact email address, formatted as a valid email string."
)

education_schema = ResponseSchema(
    name="education",
    description="A list of the candidate's academic qualifications. Each record should be an object explicitly detailing the degree obtained, the educational institution, and the graduation year."
)

skills_schema = ResponseSchema(
    name="skills",
    description="An array of strings representing the candidate's core competencies, including technical abilities, tools, languages, and professional skills."
)

experience_schema = ResponseSchema(
    name="experience",
    description="A list of the candidate's professional work history. Each record should be an object specifying the job title or role, the employing company, and the duration of employment."
)

response_schemas = [
    full_name_schema, email_schema, education_schema, skills_schema, experience_schema
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The complete legal name of the candidate, formatted as a single continuous string.
	"email": string  // The candidate's primary contact email address, formatted as a valid email string.
	"education": string  // A list of the candidate's academic qualifications. Each record should be an object explicitly detailing the degree obtained, the educational institution, and the graduation year.
	"skills": string  // An array of strings representing the candidate's core competencies, including technical abilities, tools, languages, and professional skills.
	"experience": string  // A list of the candidate's professional work history. Each record should be an object specifying the job title or role, the employing company, and the duration of employment.
}
```


In [6]:
cv_extraction_template = """
You are an expert HR assistant that extracts structured information
from a candidate's resume or CV.

Your task is to extract information from the resume below.

IMPORTANT RULES:
1. Return ONLY the final JSON object.
2. Do NOT return the schema or format instructions.
3. Do NOT return placeholder values such as "string".
4. Replace every placeholder with information extracted from the resume.
5. Use valid JSON syntax.
6. Use double quotes around all JSON keys and string values.
7. Do NOT include comments in the JSON.
8. Do NOT include any explanation before or after the JSON.

Only extract information that is explicitly present in the CV.
Do not invent, guess, or infer information.
If the CV contains no work experience, return an empty list for experience.

The required output format is:

{format_instructions}

Resume:
{cv_text}
"""

user_input = """John Smith – john.smith@email.com

Education: B.Sc. Computer Science, MIT, 2020

Skills: Python, Machine Learning, Data Analysis

Experience:

- Software Engineer at Google (2020–2023)

- Data Scientist at OpenAI (2023–Present)
"""

prompt = PromptTemplate(
    template=cv_extraction_template,
    input_variables=["cv_text", "format_instructions"]
).format(
    cv_text=user_input,
    format_instructions=format_instructions
)

response = generate_text(prompt, max_length=1000)[0]

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are an expert HR assistant that extracts structured information
from a candidate's resume or CV.

Your task is to extract information from the resume below.

IMPORTANT RULES:
1. Return ONLY the final JSON object.
2. Do NOT return the schema or format instructions.
3. Do NOT return placeholder values such as "string".
4. Replace every placeholder with information extracted from the resume.
5. Use valid JSON syntax.
6. Use double quotes around all JSON keys and string values.
7. Do NOT include comments in the JSON.
8. Do NOT include any explanation before or after the JSON.

Only extract information that is explicitly present in the CV.
Do not invent, guess, or infer information.
If the CV contains no work experience, return an empty list for experience.

The required output format is:

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The complete legal name of 

In [7]:
import re
def extract_json_block(text):
    matches = re.findall(
        r'```\s*json\s*(.*?)\s*```',
        text,
        re.DOTALL | re.IGNORECASE
    )

    if not matches:
        raise ValueError("No JSON block found")

    return matches[-1].strip()


json_text = extract_json_block(response)

output_data = output_parser.parse(json_text)

print("\n========== EXTRACTED FIELDS ==========")

print("\nName:")
print(output_data.get("full_name"))

print("\nEmail:")
print(output_data.get("email"))

print("\nSkills:")
print(output_data.get("skills"))

print("\nEducation:")
print(output_data.get("education"))

print("\nExperience:")
print(output_data.get("experience"))


========== EXTRACTED FIELDS ==========

Name:
John Smith

Email:
john.smith@email.com

Skills:
['Python', 'Machine Learning', 'Data Analysis']

Education:
[{'degree': 'B.Sc. Computer Science', 'institution': 'MIT', 'graduation_year': 2020}]

Experience:
[{'role': 'Software Engineer', 'company': 'Google', 'period': '2020–2023'}, {'role': 'Data Scientist', 'company': 'OpenAI', 'period': '2023–Present'}]


In [8]:
!pip install PyPDF2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00a 0:00:01


In [9]:
from PyPDF2 import PdfReader
from langchain_core.prompts import PromptTemplate

def extract_cv_data(pdf_path):
    reader = PdfReader(pdf_path)

    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

    prompt = PromptTemplate(
        template=cv_extraction_template,
        input_variables=["cv_text", "format_instructions"]
    ).format(
        cv_text=full_text,
        format_instructions=format_instructions
    )

    response = generate_text(prompt, max_length=2000)[0]
    json_text = extract_json_block(response)

    print("EXTRACTED JSON BLOCK:\n", repr(json_text))

    try:
        return output_parser.parse(json_text)
    except Exception:
        for line_number, line in enumerate(json_text.splitlines(), start=1):
            print(f"{line_number}: {line}")
        raise

In [10]:
NGROK_TOKEN = "3K9TIXBx5IB11ViCyCfiz99ChtJ_334XFt6WQBPjjojmqLx9K"
API_KEY = "nothing111"

In [11]:
!pip install fastapi uvicorn pyngrok

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [12]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn
import threading
import time
import socket
from pyngrok import ngrok, conf

app = FastAPI()


@app.post("/generate")
async def gen(req: Request):

    print("===== REQUEST RECEIVED =====")

    try:
        print("Authorization:", req.headers.get("authorization"))

        data = await req.json()
        print("Request data:", data)

        pdf_path = data.get("pdf_path", "")
        print("PDF path:", pdf_path)

        if not pdf_path:
            raise HTTPException(
                status_code=400,
                detail="pdf_path is required"
            )

        print("Calling extract_cv_data...")

        output_data = extract_cv_data(pdf_path)

        print("extract_cv_data() SUCCESS")
        print("Output:", output_data)

        return {
            "response": output_data
        }

    except HTTPException:
        raise

    except Exception as e:
        import traceback

        print("===== EXCEPTION =====")
        traceback.print_exc()

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port
    
port = free_port()

conf.get_default().auth_token = NGROK_TOKEN

public_url = ngrok.connect(port).public_url

print("Your public URL:", public_url)


def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )


threading.Thread(
    target=run,
    daemon=True
).start()

time.sleep(1)

print(f"API endpoint: {public_url}/generate")

Your public URL: https://economic-smith-gurgling.ngrok-free.dev                                     


INFO:     Started server process [59]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:37337 (Press CTRL+C to quit)


API endpoint: https://economic-smith-gurgling.ngrok-free.dev/generate


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


===== REQUEST RECEIVED =====
Authorization: Bearer nothing111
Request data: {'pdf_path': '/kaggle/input/datasets/omarahmed05/cvcvcv/Omar Ahmed Abd El-Kader.pdf'}
PDF path: /kaggle/input/datasets/omarahmed05/cvcvcv/Omar Ahmed Abd El-Kader.pdf
Calling extract_cv_data...
EXTRACTED JSON BLOCK:
 '{\n\t"full_name": "Omar Ahmed Abd El-Kader Ibrahim",\n\t"email": "omarabdelkader.2.05@gmail.com",\n\t"education": [\n\t\t{\n\t\t\t"degree": "Bachelor’s degree in Computer and Systems Engineering",\n\t\t\t"institution": "Ain-Shams University",\n\t\t\t"graduation_year": "2022 - 2027"\n\t\t}\n\t],\n\t"skills": [\n\t\t"RTL design basics",\n\t\t"ARM AMBA protocols",\n\t\t"SystemVerilog datatypes and threading",\n\t\t"SystemVerilog interfaces and subroutines",\n\t\t"functional coverage models",\n\t\t"SystemVerilog assertions (SVA)",\n\t\t"constrained-random stimuli",\n\t\t"UVM structures, components, sequences, and configuration",\n\t\t"UVM Phasing, TLM, and factory",\n\t\t"complex testbench verification